In [1]:
import numpy as np
import sys, os
import autograd.numpy as np
import matplotlib.pyplot as plt
import io 

**Collaborator's Data**

In [2]:
directory = './data' 

bs= []
vrels = []
labels = []
dms = []

# List all files in the directory
for filename in os.listdir(directory):
    if filename != 'summary.txt':
        print("Looking at file", filename)
        file_path = os.path.join(directory, filename)
        data = np.genfromtxt(file_path,skip_header = True, dtype = float)
        v_inf1 = data[0,24]
        v_inf2 = data[0,25]
        m_unbound = data[-1,18]

        #initial masses 
        mass2_0 = data[0,2] 
        mass1_0 = data[0,1]

        vrel = (v_inf1+v_inf2)*437.

        b = np.abs(data[0,23])

        mass2 = data[-1,2]
        mass1 = data[-1,1]

        #calculate fractional mass loss 
        dm = (m_unbound)/(mass1_0 + mass2_0)
        
        #create my labels
        # 0 complete disruption of both stars
        # 1 merged 
        # 2 unmerged 


        if mass2 == 0.0 and mass1 != 0.0:
            outcome = 1
        elif mass2 != 0.0 and mass1 != 0.0:
            outcome = 2
        elif mass2 == 0.0 and mass1 == 0.0:
            outcome = 0
        else:
            print("IDK WHAT HAPPENED HERE")
            

        bs.append(b)
        vrels.append(vrel)
        labels.append(outcome)
        dms.append(dm)

Looking at file massAndMore3.out
Looking at file massAndMore.out
Looking at file massAndMore2.out
Looking at file massAndMore_0p4_900.out
Looking at file massAndMore_0p4_2500.out
Looking at file massAndMore_rperi0_1200.out
Looking at file massAndMore_0p4_1200_new.out
Looking at file massAndMore_0p8_600.out
Looking at file massAndMore_rperi0_900.out
Looking at file massAndMore_0_600.out
Looking at file massAndMore_0p8_1700.out
Looking at file massAndMore_0p4_600.out
Looking at file massAndMore_0p8_900.out
Looking at file massAndMore_rp0p4_3300.out
Looking at file massAndMore_rperi0p8_500.out
Looking at file massAndMore_0p4_1700.out
Looking at file massAndMore2500_0p8.out
Looking at file massAndMore_rperi0p8_300.out


**Freitag and Benz Data**

In [77]:
#Code was adapted by code sent by Jamie 
bs= []
vrels = []
labels = []
dms = []

# Read and process summary.txt
with open('./data/summary.txt', 'r') as file:
    lines = file.readlines()

# Filter out lines that are just dashes
lines = [line for line in lines if not line.strip().startswith('-')]

# Convert the filtered lines to a single string and use genfromtxt
data = np.genfromtxt(io.StringIO(''.join(lines)), skip_header=52, names=True, dtype=None, encoding=None)

# Check the fields
print(data.dtype.names)

R_sun = 7e8
M_sun = 2e30
G = 6.67e-11
vunit = (G*M_sun/R_sun)**0.5 * 1e-3
print('velocity unit=',vunit,'km/s')

coll_id = data['Coll_id']
mini1 = data['Mini1']
mini2 = data['Mini2']
vrel_inf = data['Vrel_inf'] * vunit
imp_param = data['ImpParam']
mfin1 = data['Mfin1']
mfin2 = data['Mfin2']
n_star = data['Nstar']
dm = data["dM"]

# Look for places in data file where final star indices should be swapped
for i in range(len(coll_id)):
    if mfin1[i] > mfin2[i]:
        # print(f"Will swap Mfin1 and Mfin2 for Coll_id: {coll_id[i]}, Mini1: {mini1[i]}, Mini2: {mini2[i]}, Mfin1: {mfin1[i]}, M2: {mfin2[i]}")
        mfin1[i], mfin2[i] = mfin2[i], mfin1[i]

# vrel_inf = vrel_inf[(mini1 == mini2) & (mini1 <= 1.)]
# b        = imp_param[(mini1 == mini2) & (mini1 <= 1.)]
# mfin1    = mfin1[(mini1 == mini2) & (mini1 <= 1.)]
# mfin2    = mfin2[(mini1 == mini2) & (mini1 <= 1.)]
# dm       = dm[(mini1 == mini2) & (mini1 <= 1.)]
# coll_id = coll_id[(mini1 == mini2) & (mini1 <= 1.)]

vrel_inf = vrel_inf[(mini1 == mini2) & (mini1 == 1.)]
b        = imp_param[(mini1 == mini2) & (mini1 == 1.)]
mfin1    = mfin1[(mini1 == mini2) & (mini1 == 1.)]
mfin2    = mfin2[(mini1 == mini2) & (mini1 == 1.)]
dm       = dm[(mini1 == mini2) & (mini1 == 1.)]
coll_id = coll_id[(mini1 == mini2) & (mini1 == 1.)]

print(len(mfin2))

# create my labels
# 0 complete disruption of both stars
# 1 merged 
# 2 unmerged 

for i in range(len(vrel_inf)):
    if (mfin2[i] == 0.0 and mfin1[i] != 0.0) or (mfin1[i] == 0.0 and mfin2[i] != 0.0):
        outcome = 1
    elif mfin2[i] != 0.0 and mfin1[i] != 0.0:
        outcome = 2
    elif mfin2[i] == 0.0 and mfin1[i] == 0.0:
        outcome = 0
    else:
        print(mfin1[i], mfin2[i])
        print("IDK WHAT HAPPENED HERE")

    bs.append(b[i])
    vrels.append(vrel_inf[i])
    labels.append(outcome)      
    dms.append(dm[i])
    

('Coll_id', 'Mini1', 'Mini2', 'Vrel_inf', 'ImpParam', 'Mfin1', 'Mfin2', 'Nstar', 'dM', 'dE', 'dL', 'dTheta')
velocity unit= 436.5448757818932 km/s
296


In [78]:
#Append data to a file
data = np.array([bs, vrels])
data_labels = np.array([labels])

np.savetxt('data.csv', data, delimiter=',', header='b[RSUN], v_inf[km/s]', comments='', fmt='%.6f')
np.savetxt('data_labels.csv', data_labels, delimiter=',', header='Outcome', comments='')

In [5]:
#Append data to a file with the dm 
# data = np.array([bs, vrels, labels, dms,])
# np.savetxt('data_dm.csv', data, delimiter=',', header='b[RSUN], v_inf[km/s], Outcome, dM ', comments='')

In [101]:
bs = np.array(bs)
vrels = np.array(vrels)
labels = np.array(labels)
coll_id[(bs > 0.40) & (vrels > 3000) & (labels == 0)]

array([2056, 2072, 2091, 2092])

In [102]:
print(vrels[coll_id == 2056])
print(vrels[coll_id == 2056]/vunit)
print(bs[coll_id == 2056])

[3928.90388204]
[9.]
[0.406]
